In [93]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_all(seed=0):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [94]:
df = pd.read_csv('dataset_collection.csv')
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=df.iloc[:, -1]
)
train_df.to_csv('./data/train.csv', index=False)
test_df.to_csv('./data/test.csv', index=False)
print(f"Saved train.csv ({len(train_df)}) / test.csv ({len(test_df)})")

Saved train.csv (135) / test.csv (34)


In [96]:
# Cell 3: Load & Preprocess
data_train = pd.read_csv('./data/train.csv').to_numpy()
data_test  = pd.read_csv('./data/test.csv').to_numpy()

# Split features (first 240 columns) and labels (last column)
X_train = data_train[:, :-1]
y_train = data_train[:,  -1].astype(int)
X_test  = data_test[:,  :-1]
y_test  = data_test[:,   -1].astype(int)

# Convert labels from [1..8] → [0..7]
y_train -= 1
y_test  -= 1

# Standardize FEATURES only
means = X_train.mean(axis=0)
stds  = X_train.std(axis=0)
np.savez('scaler.npz', means=means, stds=stds)
X_train = (X_train - means) / stds
X_test  = (X_test  - means) / stds

# To PyTorch tensors
X_train_t = torch.from_numpy(X_train).float().to(device)
y_train_t = torch.from_numpy(y_train).long().to(device)
X_test_t  = torch.from_numpy(X_test).float().to(device)
y_test_t  = torch.from_numpy(y_test).long().to(device)

train_ds = TensorDataset(X_train_t, y_train_t)
test_ds  = TensorDataset(X_test_t,  y_test_t)
loader_train = DataLoader(train_ds, batch_size=16, shuffle=True)
loader_test  = DataLoader(test_ds,  batch_size=16, shuffle=False)

In [ ]:
# Cell 4: Define Models
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.net(x)

class LargeMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.net(x)

In [97]:
# Cell 5: Instantiate the right model
input_dim  = X_train.shape[1]      # 240
num_classes = len(np.unique(y_train))  # 8
if len(train_ds) >= 100:
    model = LargeMLP(input_dim, num_classes)
    print("→ Using LargeMLP")
else:
    model = SimpleMLP(input_dim, num_classes)
    print("→ Using SimpleMLP")

model = model.to(device)

→ Using LargeMLP


In [98]:
# Cell 6: Training & Evaluation Functions
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_epoch():
    model.train()
    total_loss = 0
    correct = 0
    for x, y in loader_train:
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        pred = out.argmax(dim=1)
        correct += (pred == y).sum().item()
    return total_loss/len(train_ds), correct/len(train_ds)

def eval_epoch():
    model.eval()
    total_loss = 0
    correct = 0
    with torch.no_grad():
        for x, y in loader_test:
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item() * x.size(0)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
    return total_loss/len(test_ds), correct/len(test_ds)

In [99]:
# Cell 7: Training Loop
epochs = 30
# --- before the loop ---
best_val_loss = float('inf')

for ep in range(1, epochs+1):
    train_loss, train_acc = train_epoch()
    val_loss,   val_acc   = eval_epoch()

    print(f"Epoch {ep:02d}: "
          f"Train Loss {train_loss:.4f}, Acc {train_acc:.3f} | "
          f"Val Loss   {val_loss:.4f}, Acc {val_acc:.3f}")

    # --- Checkpointing logic: ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  ↳ New best model (val_loss={val_loss:.4f}) saved to best_model.pth")

# --- after the loop, if you still want the final model too ---
torch.save(model.state_dict(), 'last_model.pth')
print("Final model saved to last_model.pth")



Epoch 01: Train Loss 2.0102, Acc 0.215 | Val Loss   1.9398, Acc 0.324
  ↳ New best model (val_loss=1.9398) saved to best_model.pth
Epoch 02: Train Loss 1.7676, Acc 0.348 | Val Loss   1.7884, Acc 0.353
  ↳ New best model (val_loss=1.7884) saved to best_model.pth
Epoch 03: Train Loss 1.6157, Acc 0.452 | Val Loss   1.7123, Acc 0.324
  ↳ New best model (val_loss=1.7123) saved to best_model.pth
Epoch 04: Train Loss 1.6434, Acc 0.400 | Val Loss   1.7083, Acc 0.294
  ↳ New best model (val_loss=1.7083) saved to best_model.pth
Epoch 05: Train Loss 1.5408, Acc 0.430 | Val Loss   1.6692, Acc 0.294
  ↳ New best model (val_loss=1.6692) saved to best_model.pth
Epoch 06: Train Loss 1.4634, Acc 0.474 | Val Loss   1.5991, Acc 0.265
  ↳ New best model (val_loss=1.5991) saved to best_model.pth
Epoch 07: Train Loss 1.3411, Acc 0.563 | Val Loss   1.5861, Acc 0.294
  ↳ New best model (val_loss=1.5861) saved to best_model.pth
Epoch 08: Train Loss 1.3080, Acc 0.585 | Val Loss   1.5682, Acc 0.265
  ↳ New best 

In [104]:
# 1) Rebuild the exact model class and load best_model.pth
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()

# 2) Load your scaler
scaler = np.load('scaler.npz')
means, stds = scaler['means'], scaler['stds']

# 3) Prepare your single sample (length-240 list)
raw_df = pd.read_csv('raw.csv', header=None)   # or header=0 if you have column names
raw  = raw_df.values.astype(np.float32)     # shape (N,240)

x   = (raw - means) / stds
x_t = torch.from_numpy(x).unsqueeze(0).to(device)  # shape [1,240]
x_t = x_t.to(device, dtype=torch.float32)

# 4) Predict
with torch.no_grad():
    logits = model(x_t)
    pred   = logits.argmax(dim=1).item() + 1        # map back to 1–8

print("Predicted class:", pred)


RuntimeError: running_mean should contain 1 elements not 128